# Introduction

# Stage 01 — Data and provenance

**Pipeline position:** first analysis stage. Reads nothing but production inputs; writes the eligible
row set that stages 02–07 all descend from.

## What this stage answers

Before any hit rate can mean anything, four things have to be true, and this notebook establishes
each one:

1. **The inputs are the bytes we think they are.** Every source is SHA-256'd and compared against the
   digests recorded by the original 2026-08-02 run.
2. **The predictions are genuinely out-of-sample.** The generating code is read directly and shown to
   train season *Y* on seasons `< Y` only. A saved CSV cannot demonstrate this; only its generator
   can.
3. **The join is exact.** Every walk-forward row must find its season-dataset partner, and the
   position recorded in the source file must agree with the position the dataset assigns.
4. **The population is the declared one.** QB rookies out (that arm was held back from the shipped
   surface); rows without an ADP, a prediction or an actual out. Injuries deliberately *not* filtered
   — both systems forecast season totals, so availability is part of the target.

## Inputs

| Path | Role |
|---|---|
| `fantasy/projections/results/{,wr_,te_,qb_}walkforward_predictions.csv` | predictions + observed season totals, 2021–2025 |
| `fantasy/seasonal_projections/season_dataset_2014_2025.csv` | ADP, team, identity, dataset Sleeper column |
| `fantasy/projections/build_{rb,wr,te,qb}_projection.py` | the generating code, audited statically |
| `archive/original_2026-08-02/manifest.json` | the prior run's input digests, for comparison |

## Outputs

| Path | Contents |
|---|---|
| `interim/eligible_rows.csv` | the filtered study population, one row per (season, player) |
| `interim/stage01_diagnostics.json` | input hashes, join diagnostics, exclusion counts |

## Gate

Every integrity check in this notebook is a hard assertion. If any fails, stage 02 must not run —
the failure means the study population is not what it claims to be.

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** the four walk-forward CSVs, the season dataset, the four builder scripts, and the archived manifest
**and writes:** `interim/eligible_rows.csv` and `interim/stage01_diagnostics.json`

### Explain — SHA-256 every input and compare against the archived run

Reproducibility here means more than "the code runs" — it means the numbers are pinned to specific
bytes.

**What is hashed.** The four walk-forward prediction CSVs, the season dataset supplying ADP and
identity, and the four builder scripts that generated the walk-forward files. Outputs *and* the logic
that produced them, so a future discrepancy can be attributed to changed data or changed code rather
than guessed at.

**The comparison that matters.** The five data digests are checked against
`archive/original_2026-08-02/manifest.json`. The repository was renamed `BettingEdgeContinued` →
`JoSchoAnalytics` between the two runs, so this confirms the rename moved files without altering
contents, and that every number recomputed downstream is directly comparable to the archived run.

The builders are hashed but **not** compared: the original manifest recorded only data inputs, so
there is no prior digest for them. Their logic is audited directly in the next cell instead, which is
stronger than a hash comparison would have been.

`INPUT_HASHES` is carried into `interim/stage01_diagnostics.json` and re-verified by stage 07.

In [2]:
INPUT_HASHES = {}
for label, p in {**{f"walkforward_{k}": v for k, v in WF_FILES.items()},
                 "season_dataset": SEAS_CSV,
                 **{f"builder_{k}": v for k, v in BUILDERS.items()}}.items():
    INPUT_HASHES[label] = {"path": str(p.relative_to(REPO)).replace("\\", "/"),
                           "sha256": sha256_file(p), "bytes": p.stat().st_size}

print("INPUT PROVENANCE (SHA-256)")
print("-" * 108)
for label, rec in INPUT_HASHES.items():
    print(f"  {label:18s} {rec['sha256']}  {rec['bytes']:>10,} B  {rec['path']}")

_orig = json.loads((ARCHIVE / "manifest.json").read_text(encoding="utf-8"))
_orig_hashes = {k.replace("\\", "/"): v["sha256"] for k, v in _orig["inputs"].items()}
_by_path = {r["path"]: r["sha256"] for r in INPUT_HASHES.values()}

print("\nCOMPARISON vs the archived 2026-08-02 run "
      "(repo renamed BettingEdgeContinued -> JoSchoAnalytics in between)")
print("-" * 108)
_drift = [p for p, old in _orig_hashes.items() if _by_path.get(p) != old]
for path, old in _orig_hashes.items():
    print(f"  {'MATCH   ' if _by_path.get(path) == old else 'DRIFTED '} {path}")
print(f"\ndata inputs compared: {len(_orig_hashes)} | identical: {len(_orig_hashes)-len(_drift)} "
      f"| drifted: {len(_drift)}")
assert not _drift, f"source data changed since the archived run: {_drift}"
print("=> every data input is byte-identical to the archived run.")

INPUT PROVENANCE (SHA-256)
------------------------------------------------------------------------------------------------------------
  walkforward_RB     c8c0e1584a18452adcb2c3510b1ff91104b8f44e150a0a89b46e21f6a9f04411      49,283 B  fantasy/projections/results/walkforward_predictions.csv
  walkforward_WR     49f6b0d69f796d90c1fe5b3bd9a1fdd0414f36d023a0debe056308ca0d3957db      75,912 B  fantasy/projections/results/wr_walkforward_predictions.csv
  walkforward_TE     ab663974a8888334ee7fd7cf69c893aa5b60cd08e30df31642677e8dee2c8160      39,995 B  fantasy/projections/results/te_walkforward_predictions.csv
  walkforward_QB     b2545b0c55ab3dfbd0ff378b60d60a4ab0e6e16eee283ef30a7e690b2fb33acc      26,699 B  fantasy/projections/results/qb_walkforward_predictions.csv
  season_dataset     dfdf38d9372c830b9fdfffe914024000409e08f01f21536f81876c199e0e2319   2,758,268 B  fantasy/seasonal_projections/season_dataset_2014_2025.csv
  builder_RB         7ffb77d4db746f3ebbc6c2ecb475481b05ef59a46a53063

### Interpretation — the data is byte-identical to the original run

**5 data inputs compared, 5 identical, 0 drifted.** The repository rename moved files without touching
contents.

That is the precondition that makes this pipeline a genuine *reproduction* rather than a fresh
analysis that happens to resemble the old one. Any number produced downstream that differs from the
archived report cannot be blamed on changed data — it would have to be a change in this pipeline's
logic, and would need explaining rather than accepting. Stage 07 confirms that nothing differs.

The builder hashes are recorded as a baseline for future runs. They are deliberately not compared,
because the original manifest never captured them; the next cell audits their *behaviour* instead,
which is what actually matters for validity.

`INPUT_HASHES` now holds the provenance baseline for the whole pipeline.

### Explain — audit the generator, do not trust the CSV

The saved CSVs contain *outputs*. The claim this entire study depends on — that the prediction for
season *Y* came from a model trained only on seasons before *Y* — lives in the code that wrote them.
A leaked prediction would make every hit rate downstream meaningless, so this cell reads the
generating source and shows the guarantee rather than assuming it.

**What it prints and checks.**

1. The body of `walk_forward()` from `build_rb_projection.py`, the function that produced every row in
   all four CSVs. The lines to read are the training slice `df[df.season < Y]` and the assertion
   `assert (tr.season < Y).all()`, which fails the build outright on a leak. Both are verified
   textually.
2. That `build_{wr,te,qb}_projection.py` each **import** that same function rather than defining
   their own — so the guarantee is shared, not duplicated four times with three chances to drift.
3. That the builders' own `TEST_SEASONS` constant matches this pipeline's window, confirming the
   panel was not chosen here after the fact.

This is a **static** audit. It reads source; it does not re-run training, which is explicitly out of
scope. It establishes the specific claim the study needs — that a prediction for season Y is
out-of-sample with respect to season Y — and nothing broader.

In [3]:
_rb_src = BUILDERS["RB"].read_text(encoding="utf-8")
_m = re.search(r"^def walk_forward\(.*?(?=\n\ndef |\n\n# ---)", _rb_src, re.S | re.M)
assert _m, "walk_forward() not found in the RB engine"
_wf_src = _m.group(0)

print("GENERATOR AUDIT — fantasy/projections/build_rb_projection.py :: walk_forward()")
print("=" * 100)
for i, line in enumerate(_wf_src.splitlines(), 1):
    mark = " <<<" if ("season < Y" in line or "WALK-FORWARD LEAK" in line) else ""
    print(f"  {i:>3} | {line}{mark}")
print("=" * 100)

_train_slice = "df[(df.season < Y)]" in _wf_src
_leak_assert = 'assert (tr.season < Y).all(), f"WALK-FORWARD LEAK' in _wf_src
print(f"training slice restricted to seasons < Y : {_train_slice}")
print(f"explicit leak assertion present          : {_leak_assert}")
assert _train_slice and _leak_assert, "the walk-forward leak guarantee is not present in the engine"

print("\nENGINE REUSE — do the other three positions share this exact function?")
print("-" * 100)
for pos in ("WR", "TE", "QB"):
    src = BUILDERS[pos].read_text(encoding="utf-8")
    imports_engine = bool(re.search(r"from build_rb_projection import\b", src))
    imports_wf = ("walk_forward" in re.search(r"from build_rb_projection import \(([^)]*)\)",
                                              src, re.S).group(1)) if imports_engine else False
    defines_own = bool(re.search(r"^def walk_forward\(", src, re.M))
    print(f"  {pos}: imports RB engine={imports_engine} | imports walk_forward={imports_wf} "
          f"| defines its own={defines_own}")
    assert imports_engine and imports_wf and not defines_own, f"{pos} does not share the audited engine"

_ts = re.search(r"^TEST_SEASONS = (\[[0-9, ]+\])", _rb_src, re.M).group(1)
print(f"\nbuilders' own TEST_SEASONS constant : {_ts}")
assert eval(_ts) == TEST_SEASONS, "pipeline season window disagrees with the builders'"
print("=> every walk-forward row for season Y came from a model fit on seasons < Y only, enforced in code.")

GENERATOR AUDIT — fantasy/projections/build_rb_projection.py :: walk_forward()
    1 | def walk_forward(df, feats, tag):
    2 |     """Per prereg §8: for each Y in 2021-2025, inner-CV select on seasons<Y, fit on seasons<Y, predict Y."""
    3 |     rows, chosen = [], []
    4 |     for Y in TEST_SEASONS:
    5 |         tr = df[(df.season < Y)].dropna(subset=["y"]) <<<
    6 |         te = df[df.season == Y].dropna(subset=["y"])
    7 |         if len(tr) < 60 or len(te) == 0:
    8 |             continue
    9 |         assert (tr.season < Y).all(), f"WALK-FORWARD LEAK ({tag}, {Y})" <<<
   10 |         t0 = time.time()
   11 |         (fam, params, imae), per_family = nested_select(tr, feats)
   12 |         Xtr, Xte = _prep(fam, tr, te, feats)
   13 |         p = _fit_predict(fam, params, Xtr, tr["y"].to_numpy(float), Xte)
   14 |         rows.append(pd.DataFrame({"season": Y, "player_id": te["player_id"].values,
   15 |                                   "player": te["player"].value

### Interpretation — the leak guarantee is in the code, not just the documentation

The printed function body settles the central validity question. Line 5 builds the training frame as
`df[(df.season < Y)]` and line 9 asserts `(tr.season < Y).all()` with the message
`WALK-FORWARD LEAK`. Both textual checks returned `True`, so a violation would have raised at build
time rather than returning a quietly contaminated number.

The reuse check is what extends this from one position to four: WR, TE and QB each import
`walk_forward` from the RB engine and **none defines its own**. One implementation, one guard, no
opportunity for three copies to drift apart. The builders' `TEST_SEASONS` is
`[2021, 2022, 2023, 2024, 2025]`, matching this pipeline exactly — so the panel is the builders' own,
not something imposed here after seeing results.

What this does **not** establish: anything about whether the *features* are free of look-ahead. That
is the projection pre-registration's problem, not this study's. This establishes only that a
prediction for season Y is out-of-sample with respect to season Y — which is precisely the assumption
every hit rate downstream rests on.

With the generator trusted, the CSVs can be loaded as evidence.

### Explain — load the four walk-forward files and check each one individually

Each position's file is loaded and validated **before** anything is concatenated, so a problem is
attributed to the right file rather than surfacing later as an anonymous anomaly.

**Schema.** `season`, `grp` (`vet`/`rook`), `player_id`, `player`, `y` (observed season-total
half-PPR), `pred` (the walk-forward prediction), `sleeper` (Sleeper's preseason projection as
recorded at build time), `model` (the family the inner CV selected for that fold).

**Per-file assertions:** the schema is complete; `(season, player_id)` is unique (a duplicate would
double-count a player); every season lies inside 2021–2025 (a stray season would mean a file from a
different run); `y` and `pred` are fully populated (a missing prediction cannot be ranked).

**Output.** One row per position — count, season coverage, veteran/rookie split, Sleeper coverage,
and the model families chosen — then the concatenated frame with `file_position` recorded so the next
cell can verify the source file agrees with the dataset's position assignment.

**On the `sleeper` column.** It is taken from the walk-forward file rather than the season dataset —
the value as of the build. The next cell measures the difference between the two vintages rather than
silently preferring one.

In [4]:
_frames, _rows = [], []
for pos, path in WF_FILES.items():
    d = pd.read_csv(path)
    expected = {"season", "grp", "player_id", "player", "y", "pred", "sleeper", "model"}
    assert expected <= set(d.columns), f"{path.name}: missing {expected - set(d.columns)}"
    assert not d.duplicated(["season", "player_id"]).any(), f"{path.name}: duplicate key"
    assert set(d["season"]) <= set(TEST_SEASONS), f"{path.name}: season outside {TEST_SEASONS}"
    assert d["y"].notna().all() and d["pred"].notna().all(), f"{path.name}: null y or pred"
    d["file_position"] = pos
    _frames.append(d)
    _rows.append({"position": pos, "rows": len(d),
                  "seasons": f"{d.season.min()}-{d.season.max()}", "n_seasons": d.season.nunique(),
                  "veteran": int((d.grp == "vet").sum()), "rookie": int((d.grp == "rook").sum()),
                  "sleeper_present": int(d.sleeper.notna().sum()),
                  "sleeper_cov": f"{d.sleeper.notna().mean():.1%}",
                  "model_families": ",".join(sorted(d.model.dropna().unique()))})

WF = pd.concat(_frames, ignore_index=True)
assert not WF.duplicated(["season", "player_id", "file_position"]).any()

print("WALK-FORWARD FILES — per-position load and integrity")
print(pd.DataFrame(_rows).to_string(index=False))
print(f"\nconcatenated: {len(WF):,} rows x {WF.shape[1]} columns")
print(f"duplicate (season, player_id) across positions: {int(WF.duplicated(['season','player_id']).sum())}")

WALK-FORWARD FILES — per-position load and integrity
position  rows   seasons  n_seasons  veteran  rookie  sleeper_present sleeper_cov      model_families
      RB   802 2021-2025          5      645     157              486       60.6%            lightgbm
      WR  1242 2021-2025          5     1006     236              657       52.9% elasticnet,lightgbm
      TE   677 2021-2025          5      558     119              304       44.9%            lightgbm
      QB   430 2021-2025          5      380      50              262       60.9%    lightgbm,xgboost

concatenated: 3,151 rows x 9 columns
duplicate (season, player_id) across positions: 0


### Interpretation — 3,151 clean rows, and a Sleeper-coverage warning

All four files loaded with unique keys, seasons confined to 2021–2025, and no null predictions or
actuals: **802 RB + 1,242 WR + 677 TE + 430 QB = 3,151 rows**, zero duplicate keys across positions.

**The column that matters most here is Sleeper coverage, and it is far from complete**: RB 60.6%, WR
52.9%, QB 60.9%, and TE only **44.9%**. Barely more than half the raw population can express a
Sleeper-vs-ADP disagreement at all. That is the entire reason this study needs two rank universes —
A leaves those rows in the rank denominators, B removes them — and it is why TE will contribute fewer
agreement calls than its row count suggests. (Coverage recovers to 89.2% after the ADP filter two
cells below, because Sleeper-less rows are disproportionately players with no ADP either.)

`model_families` is an incidental but useful confirmation that these are genuine nested-CV outputs:
RB and TE selected LightGBM in every fold, while **WR mixed ElasticNet and LightGBM** and **QB mixed
LightGBM and XGBoost**. A single family everywhere would have hinted at a hard-coded choice; the
variation is consistent with the inner CV actually selecting per fold.

### Explain — join to the season dataset and prove the join is exact

The walk-forward files carry predictions and outcomes but not the market price. ADP, team and
identity come from `season_dataset_2014_2025.csv`, joined on `(season, player_id)` with
`validate="one_to_one"`.

**Integrity assertions — any failure stops the pipeline.**

- the season dataset itself has unique `(season, player_id)`;
- **no walk-forward row fails to join** — an unmatched row would silently vanish from every
  denominator;
- **the joined position equals the source file's position for every row** — this is what makes the
  later "ranks are always within season-position" claim verifiable rather than assumed.

**The Sleeper vintage check.** The dataset was rebuilt 2026-07-26; the walk-forward files were written
2026-07-21. The disagreement is measured in three parts: rows where only the walk-forward file has a
value, rows where only the dataset does, and the maximum absolute difference where both do. This
study uses the walk-forward column throughout, so dataset-only rows drop out of signal evaluation —
but the size of that set is measured and reported rather than assumed negligible.

In [5]:
SD = pd.read_csv(SEAS_CSV)
assert not SD.duplicated(["season", "player_id"]).any(), "season dataset: duplicate key"

_keep = ["season", "player_id", "position", "team", "adp_half_ppr", "adp_overall_rank",
         "adp_pos_rank", "sleeper_pts_half_ppr", "is_rookie", "norm_name"]
M = WF.merge(SD[_keep], on=["season", "player_id"], how="left", validate="one_to_one")

_unjoined = int(M["position"].isna().sum())
assert _unjoined == 0, f"{_unjoined} walk-forward rows failed the join"
_mismatch = int((M["position"] != M["file_position"]).sum())
assert _mismatch == 0, f"{_mismatch} rows: joined position disagrees with the source file"

M = M.rename(columns={"position": "pos"}).drop(columns=["file_position"])
M["group"] = np.where(M["grp"] == "rook", "rookie", "veteran")

_both = M.dropna(subset=["sleeper", "sleeper_pts_half_ppr"])
JOIN_DIAG = {"season_dataset_rows": int(len(SD)), "walkforward_rows": int(len(WF)),
             "joined": int(len(M)), "unjoined": _unjoined, "position_mismatch": _mismatch,
             "adp_present": int(M.adp_half_ppr.notna().sum()),
             "sleeper_in_walkforward_only": int((M.sleeper.notna() & M.sleeper_pts_half_ppr.isna()).sum()),
             "sleeper_in_dataset_only": int((M.sleeper.isna() & M.sleeper_pts_half_ppr.notna()).sum()),
             "sleeper_value_max_abs_diff": float((_both.sleeper - _both.sleeper_pts_half_ppr).abs().max())}

print("JOIN INTEGRITY — walk-forward -> season_dataset_2014_2025.csv on (season, player_id)")
print("-" * 92)
for k, v in JOIN_DIAG.items():
    print(f"  {k:30s} {v}")

print("\nSLEEPER VINTAGE — walk-forward files (2026-07-21) vs season dataset (rebuilt 2026-07-26)")
print("-" * 92)
_only_ds = M[M.sleeper.isna() & M.sleeper_pts_half_ppr.notna()]
print(f"  rows where ONLY the rebuilt dataset carries a Sleeper value: {len(_only_ds)}")
if len(_only_ds):
    print(_only_ds[["season", "pos", "player", "sleeper_pts_half_ppr", "adp_half_ppr"]]
          .sort_values(["season", "pos"]).to_string(index=False))
print(f"\n  max |difference| where BOTH carry a value: {JOIN_DIAG['sleeper_value_max_abs_diff']}")

JOIN INTEGRITY — walk-forward -> season_dataset_2014_2025.csv on (season, player_id)
--------------------------------------------------------------------------------------------
  season_dataset_rows            7350
  walkforward_rows               3151
  joined                         3151
  unjoined                       0
  position_mismatch              0
  adp_present                    1915
  sleeper_in_walkforward_only    0
  sleeper_in_dataset_only        19
  sleeper_value_max_abs_diff     0.0

SLEEPER VINTAGE — walk-forward files (2026-07-21) vs season dataset (rebuilt 2026-07-26)
--------------------------------------------------------------------------------------------
  rows where ONLY the rebuilt dataset carries a Sleeper value: 19
 season pos           player  sleeper_pts_half_ppr  adp_half_ppr
   2021  RB     Nyheim Hines                  91.1         127.9
   2021  RB Kenneth Gainwell                  97.5         169.4
   2021  WR      Will Fuller                 156

### Interpretation — the join is exact, and the vintage gap is 19 rows in the benign direction

All three assertions passed on the full 3,151 rows: **0 unjoined, 0 position mismatches**. Every
walk-forward row found its partner, and the position in the source file agrees with the dataset's
assignment for every single row — so the "ranks are within season-position" claim is verified, not
assumed.

**1,915 of 3,151 rows carry an ADP** (about 61%). The rest are players the models scored but the
market never priced; they leave in the next cell.

The vintage difference is smaller and more benign than it could have been. Where both sources carry a
value the **maximum absolute difference is exactly 0.0** — the rebuild revised nothing. The only
asymmetry is **19 rows the rebuilt dataset gained**, with **0 in the opposite direction**, and the
names make the mechanism obvious: Nyheim Hines, Kenneth Gainwell, Josh Palmer, Will Fuller, Scott
Miller, Andrew Ogletree, Mitchell Tinsley — players whose identity resolution changed between builds.

That is about 1% of the eligible population, and the 2024–25 entries are deep (ADP 480–684). Using
the build-time column means they sit in rank denominators without expressing a disagreement.
Preferring the dataset column would add at most 19 mostly-undrafted rows — which, given what stage 03
finds about undrafted rows, would push the headline in the **flattering** direction. Using the
build-time column is the conservative choice, and it is the one taken.

### Explain — apply the population filter and write the stage handoff

Two filters define the study population, both stated in the brief rather than chosen here.

**1. QB rookies are removed.** That arm was fitted and then *held back* from the shipped surface —
`qb_rookie_board_projection.csv` is deliberately header-only — because on 7–13 rookie QBs a season it
mostly re-ranked by draft capital and projected full starter seasons for quarterbacks the market
expected to sit. Scoring a signal the product does not ship would inflate the study with predictions
nobody can act on. RB, WR and TE rookie rows are kept, because those arms did ship.

**2. A row must carry an ADP, a prediction and an actual.** No price means no disagreement to
measure; no outcome means nothing to grade.

Deliberately **not** filtered: injuries and games played. Both systems forecast *season totals*, so
availability is part of the quantity being predicted. Removing injured seasons would grade the
projections on a question neither was asked.

**Output.** `interim/eligible_rows.csv` — the population every later stage descends from — plus
`interim/stage01_diagnostics.json` carrying the input hashes, join diagnostics and exclusion counts
into the manifest that stage 07 assembles.

In [6]:
_qb_rookie = M["pos"].eq("QB") & M["grp"].eq("rook")
N_QB_ROOKIE_DROPPED = int(_qb_rookie.sum())
ELIGIBLE = M[~_qb_rookie].copy()
assert not (ELIGIBLE["pos"].eq("QB") & ELIGIBLE["grp"].eq("rook")).any(), "QB rookie survived the filter"
_pre_adp = len(ELIGIBLE)
ELIGIBLE = ELIGIBLE[ELIGIBLE.adp_half_ppr.notna() & ELIGIBLE.pred.notna() & ELIGIBLE.y.notna()].copy()

print("POPULATION FILTER")
print("-" * 88)
print(f"  joined walk-forward rows                     : {len(M):,}")
print(f"  minus QB rookie rows (arm held from shipping): -{N_QB_ROOKIE_DROPPED}")
print(f"  remaining                                    : {_pre_adp:,}")
print(f"  minus rows with no ADP / pred / actual       : -{_pre_adp - len(ELIGIBLE):,}")
print(f"  ELIGIBLE                                     : {len(ELIGIBLE):,}")

print("\nADP-bearing model population, rows per season x position:")
_a = ELIGIBLE.pivot_table(index="season", columns="pos", values="player_id", aggfunc="count")
_a["TOTAL"] = _a.sum(axis=1); print(_a)
print("\n... of which also carry a Sleeper projection:")
_b = ELIGIBLE[ELIGIBLE.sleeper.notna()].pivot_table(index="season", columns="pos",
                                                    values="player_id", aggfunc="count")
_b["TOTAL"] = _b.sum(axis=1); print(_b)

_n_draft = int((ELIGIBLE.adp_overall_rank <= DRAFTABLE_POOL_SIZE).sum())
print(f"\nSleeper coverage over eligible : {ELIGIBLE.sleeper.notna().sum():,}/{len(ELIGIBLE):,} "
      f"= {ELIGIBLE.sleeper.notna().mean():.1%}")
print(f"veteran / rookie               : {int((ELIGIBLE.group=='veteran').sum()):,} / "
      f"{int((ELIGIBLE.group=='rookie').sum()):,}")
print(f"drafted (adp_overall_rank<={DRAFTABLE_POOL_SIZE})   : {_n_draft:,} of {len(ELIGIBLE):,} "
      f"= {_n_draft/len(ELIGIBLE):.1%}")

_cols = ["season", "player_id", "player", "pos", "team", "grp", "group", "is_rookie", "norm_name",
         "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "pred", "sleeper", "y", "model"]
ELIGIBLE[_cols].to_csv(INTERIM / "eligible_rows.csv", index=False)
(INTERIM / "stage01_diagnostics.json").write_text(json.dumps({
    "input_hashes": INPUT_HASHES, "join_diagnostics": JOIN_DIAG,
    "qb_rookie_rows_dropped": N_QB_ROOKIE_DROPPED,
    "eligible_rows": int(len(ELIGIBLE)),
    "sleeper_covered": int(ELIGIBLE.sleeper.notna().sum()),
    "drafted_top180_rows": _n_draft,
}, indent=2), encoding="utf-8")
print(f"\nwrote interim/eligible_rows.csv ({len(ELIGIBLE):,} rows x {len(_cols)} cols)")
print(f"wrote interim/stage01_diagnostics.json")

POPULATION FILTER
----------------------------------------------------------------------------------------
  joined walk-forward rows                     : 3,151
  minus QB rookie rows (arm held from shipping): -50
  remaining                                    : 3,101
  minus rows with no ADP / pred / actual       : -1,218
  ELIGIBLE                                     : 1,883

ADP-bearing model population, rows per season x position:
pos     QB   RB   TE   WR  TOTAL
season                          
2021    50   97   44  112    303
2022    46   83   44  106    279
2023    40   92   58  119    309
2024    62  121   82  179    444
2025    68  140  125  215    548

... of which also carry a Sleeper projection:
pos     QB   RB   TE   WR  TOTAL
season                          
2021    38   87   42  103    270
2022    39   79   40  101    259
2023    36   87   51  110    284
2024    58  114   71  162    405
2025    61  119  100  181    461

Sleeper coverage over eligible : 1,679/1,883 = 89.

### Interpretation — 1,883 eligible rows, and less than half are actually drafted

The filter chain: 3,151 joined rows, minus **50 QB rookie rows**, minus 1,218 lacking an ADP, a
prediction or an actual, leaving **1,883 eligible**. The QB-rookie assertion passed, so none survived.

Two features of the season x position table drive everything downstream.

**The panel is heavily weighted toward recent seasons.** 2021 contributes 303 eligible rows and 2025
contributes **548** — an 81% increase, mostly WR (112 → 215) and TE (44 → 125). So the pooled
2021–2025 panel is not five equal seasons; it is dominated by 2024–2025, and the primary panel is 992
of the 1,883 rows. This is growth in *ADP coverage of deep players*, not growth in the NFL, and it is
the mechanism that makes the undrafted tail so influential in the recent panels specifically.

**Sleeper coverage recovers to 89.2%** (1,679 of 1,883), far better than the 45–61% in the raw files.
So universes A and B will differ by only 204 rows and the sensitivity between them should be modest —
which stage 03 confirms.

**The drafted subset is 886 of 1,883 — 47.0%.** More than half the "ADP-bearing" population sits
outside the top 180 picks. **That single number is the setup for stage 03**: if the agreement cell
concentrates in that outside-the-draft half, the headline is not measuring what it appears to.

`interim/eligible_rows.csv` is written. Stage 02 takes it from here.

# Conclusion and Next Steps

## What this stage established

**Provenance.** All five data inputs are byte-identical to the digests recorded by the original
2026-08-02 run — 5 compared, 5 identical, 0 drifted — despite the repository having been renamed
`BettingEdgeContinued` → `JoSchoAnalytics` in between. Any difference in a downstream number
therefore cannot be blamed on changed data.

**The out-of-sample guarantee.** `walk_forward()` builds its training frame as `df[df.season < Y]`
and asserts `(tr.season < Y).all()` with a `WALK-FORWARD LEAK` message. All three sibling builders
import that same function and none defines its own, so one guard covers all four positions.

**The join.** 3,151 walk-forward rows, **0 unjoined and 0 position mismatches**. The Sleeper vintage
difference is 19 rows the rebuilt dataset gained and **0 the other way**, with a maximum value
difference of exactly 0.0 where both sources carry a projection.

**The population.** 3,151 → minus 50 QB rookie rows → minus 1,218 rows lacking ADP, prediction or
actual → **1,883 eligible rows**, of which 1,679 (89.2%) carry a Sleeper projection and **886 (47%)
sit inside the draftable top 180**.

## What is now true

`interim/eligible_rows.csv` holds the study population. Every number in stages 02–07 descends from
exactly these 1,883 rows, and the diagnostics that justify them are in
`interim/stage01_diagnostics.json`.

## The number to carry forward

**Less than half the ADP-bearing population is actually drafted.** That single fact is what stage 03
turns into the study's central finding. Hold it in mind.

## Next step

Run **`02_ranks_and_signals.ipynb`**, which reads `interim/eligible_rows.csv`, explains why the
stored `adp_pos_rank` cannot be used, builds both rank universes across both populations, and derives
the agreement signal.

**Condition:** every assertion above must have passed. They did.